In [36]:
import pandas as pd
from google.cloud import bigquery


PROJECT_ID = "intrepid-hour-272417"
DATASET_ID = "kenya_employee_analytics"
TABLE_ID = "gold_employee_detail"

client = bigquery.Client(project=PROJECT_ID)

query = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
"""

df = client.query(query).to_dataframe()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Rows: 420
Columns: ['employee_id', 'full_name', 'gender', 'department', 'department_reference', 'job_title', 'county', 'hire_date', 'salary_kes', 'age', 'performance_rating']


In [37]:
df[[
    "salary_kes",
    "age",
    "performance_rating"
]].describe()

,salary_kes,age,performance_rating
count,420.0,420.0,420.0
mean,190279.761905,37.414286,3.428571
std,107134.251847,7.46542,1.193516
min,41000.0,24.0,1.0
25%,103875.0,32.0,3.0
50%,166000.0,37.0,4.0
75%,254000.0,42.25,4.0
max,585000.0,59.0,5.0


In [38]:
# Q1–Q3: Workforce and salary structure

print("=== Q1: WORKFORCE BY DEPARTMENT ===")

department_workforce = (
    df.groupby("department")
      .agg(
          employees=("employee_id", "count"),
          average_salary=("salary_kes", "mean"),
          median_salary=("salary_kes", "median")
      )
      .sort_values("employees", ascending=False)
)

department_workforce["workforce_pct"] = (
    department_workforce["employees"] / len(df) * 100
)

display(department_workforce)


print("\n=== Q2: OVERALL SALARY STRUCTURE ===")

salary_summary = df["salary_kes"].describe()

display(salary_summary)


print("\n=== Q3: SALARY BY DEPARTMENT ===")

department_salary = (
    df.groupby("department")["salary_kes"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_values("mean", ascending=False)
)

display(department_salary)

=== Q1: WORKFORCE BY DEPARTMENT ===


,employees,average_salary,median_salary,workforce_pct
department,,,,
Engineering,103,268800.970874,253000.0,24.52381
Sales,76,173967.105263,153250.0,18.095238
Operations,60,144583.333333,101000.0,14.285714
Customer Support,46,86119.565217,76500.0,10.952381
Data,45,211933.333333,204000.0,10.714286
Marketing,38,159171.052632,132250.0,9.047619
Finance,32,246296.875,205500.0,7.619048
Human Resources,20,145300.0,110500.0,4.761905



=== Q2: OVERALL SALARY STRUCTURE ===


count            420.0
mean     190279.761905
std      107134.251847
min            41000.0
25%           103875.0
50%           166000.0
75%           254000.0
max           585000.0
Name: salary_kes, dtype: Float64


=== Q3: SALARY BY DEPARTMENT ===


,count,mean,median,min,max
department,,,,,
Engineering,103,268800.970874,253000.0,87500,585000
Finance,32,246296.875,205500.0,75000,540000
Data,45,211933.333333,204000.0,109000,426000
Sales,76,173967.105263,153250.0,49500,363000
Marketing,38,159171.052632,132250.0,57500,331000
Human Resources,20,145300.0,110500.0,87500,327000
Operations,60,144583.333333,101000.0,50000,358500
Customer Support,46,86119.565217,76500.0,41000,155500


In [39]:
# Q4: Salary by job title

job_salary = (
    df.groupby("job_title")
      .agg(
          employees=("employee_id", "count"),
          average_salary=("salary_kes", "mean"),
          median_salary=("salary_kes", "median"),
          min_salary=("salary_kes", "min"),
          max_salary=("salary_kes", "max")
      )
      .sort_values("average_salary", ascending=False)
)

job_salary

,employees,average_salary,median_salary,min_salary,max_salary
job_title,,,,,
Senior Software Engineer,26,406211.538462,404000.0,292500,585000
Finance Manager,14,371285.714286,381500.0,243000,540000
Data Scientist,13,300192.307692,297500.0,221500,426000
Sales Manager,23,281586.956522,279500.0,215500,363000
Data Engineer,20,278800.0,277250.0,175500,358500
Operations Manager,19,268578.947368,257500.0,178500,358500
HR Manager,5,257200.0,259500.0,210500,327000
DevOps Engineer,20,249575.0,247500.0,157000,344000
Brand Manager,15,245666.666667,240500.0,176000,331000


In [40]:
# Q5: Does performance relate to salary?

performance_salary = (
    df.groupby("performance_rating")
      .agg(
          employees=("employee_id", "count"),
          average_salary=("salary_kes", "mean"),
          median_salary=("salary_kes", "median")
      )
      .sort_index()
)

performance_salary

,employees,average_salary,median_salary
performance_rating,,,
1,39,193282.051282,180500.0
2,46,166097.826087,142000.0
3,115,194882.608696,172000.0
4,136,191738.970588,169250.0
5,84,193464.285714,169000.0


In [41]:
from scipy.stats import spearmanr

rho, p_value = spearmanr(
    df["performance_rating"],
    df["salary_kes"]
)

print(f"Spearman correlation coefficient (ρ): {rho:.3f}")
print(f"P-value: {p_value:.4f}")

Spearman correlation coefficient (ρ): 0.033
P-value: 0.4962


In [42]:
# Q7: Salary by gender

gender_salary = (
    df.groupby("gender")
      .agg(
          employees=("employee_id", "count"),
          average_salary=("salary_kes", "mean"),
          median_salary=("salary_kes", "median")
      )
)

gender_salary

,employees,average_salary,median_salary
gender,,,
Female,210,195380.952381,169250.0
Male,210,185178.571429,160000.0


In [43]:
# Q7: Statistical test for gender salary difference

from scipy.stats import ttest_ind

female_salary = df.loc[
    df["gender"] == "Female", "salary_kes"
]

male_salary = df.loc[
    df["gender"] == "Male", "salary_kes"
]

t_stat, p_value = ttest_ind(
    female_salary,
    male_salary,
    equal_var=False
)

print(f"Mean salary difference: {female_salary.mean() - male_salary.mean():,.2f} KES")
print(f"t-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.4f}")

Mean salary difference: 10,202.38 KES
t-statistic: 0.976
P-value: 0.3297


In [44]:
# Q8: Age vs salary correlation

from scipy.stats import pearsonr

r, p_value = pearsonr(
    df["age"],
    df["salary_kes"]
)

print(f"Pearson correlation coefficient (r): {r:.3f}")
print(f"P-value: {p_value:.4f}")

Pearson correlation coefficient (r): 0.069
P-value: 0.1565


In [45]:
# Q9: Do salaries differ across departments?

from scipy.stats import kruskal

department_groups = [
    group["salary_kes"].values
    for _, group in df.groupby("department")
]

statistic, p_value = kruskal(*department_groups)

print(f"Kruskal-Wallis statistic: {statistic:.3f}")
print(f"P-value: {p_value:.4f}")

Kruskal-Wallis statistic: 147.564
P-value: 0.0000


In [46]:
# Q10: Post-hoc pairwise department comparisons

import scikit_posthocs as sp

posthoc = sp.posthoc_dunn(
    df,
    val_col="salary_kes",
    group_col="department",
    p_adjust="bonferroni"
)

display(posthoc)

,Customer Support,Data,Engineering,Finance,Human Resources,Marketing,Operations,Sales
Customer Support,1.000000e+00,1.205092e-11,2.631199e-24,7.532504e-11,0.140270,0.001165,3.054379e-02,4.399044e-07
Data,1.205092e-11,1.000000e+00,1.000000e+00,1.000000e+00,0.120138,0.135446,2.315593e-04,3.870806e-01
Engineering,2.631199e-24,1.000000e+00,1.000000e+00,1.000000e+00,0.000159,0.000011,1.586754e-12,2.905564e-06
Finance,7.532504e-11,1.000000e+00,1.000000e+00,1.000000e+00,0.073069,0.084819,2.639560e-04,2.411256e-01
Human Resources,1.402703e-01,1.201376e-01,1.588256e-04,7.306948e-02,1.000000,1.000000,1.000000e+00,1.000000e+00
Marketing,1.165010e-03,1.354457e-01,1.117016e-05,8.481881e-02,1.000000,1.000000,1.000000e+00,1.000000e+00
Operations,3.054379e-02,2.315593e-04,1.586754e-12,2.639560e-04,1.000000,1.000000,1.000000e+00,4.474668e-01
Sales,4.399044e-07,3.870806e-01,2.905564e-06,2.411256e-01,1.000000,1.000000,4.474668e-01,1.000000e+00


In [47]:
# Q11: Salary dispersion

mean_salary = df["salary_kes"].mean()
std_salary = df["salary_kes"].std()

coefficient_of_variation = std_salary / mean_salary

print(f"Mean salary: {mean_salary:,.2f} KES")
print(f"Standard deviation: {std_salary:,.2f} KES")
print(f"Coefficient of variation: {coefficient_of_variation:.3f}")

Mean salary: 190,279.76 KES
Standard deviation: 107,134.25 KES
Coefficient of variation: 0.563


In [48]:
# Store our statistical test results clearly

from scipy.stats import pearsonr, spearmanr, ttest_ind, kruskal

# Age vs salary
age_r, age_p = pearsonr(
    df["age"],
    df["salary_kes"]
)

# Performance vs salary
performance_rho, performance_p = spearmanr(
    df["performance_rating"],
    df["salary_kes"]
)

# Gender salary difference
female_salary = df.loc[
    df["gender"] == "Female", "salary_kes"
]

male_salary = df.loc[
    df["gender"] == "Male", "salary_kes"
]

gender_t, gender_p = ttest_ind(
    female_salary,
    male_salary,
    equal_var=False
)

# Department salary differences
department_groups = [
    group["salary_kes"].values
    for _, group in df.groupby("department")
]

department_h, department_p = kruskal(
    *department_groups
)

print("Results stored successfully.")

Results stored successfully.


In [49]:
print("=== FINAL STATISTICAL SUMMARY ===\n")

print("WORKFORCE")
print(f"Employees: {len(df):,}")
print(f"Departments: {df['department'].nunique()}")
print(f"Job titles: {df['job_title'].nunique()}")

print("\nSALARY")
print(f"Mean: {df['salary_kes'].mean():,.2f} KES")
print(f"Median: {df['salary_kes'].median():,.2f} KES")
print(f"Standard deviation: {df['salary_kes'].std():,.2f} KES")
print(f"Minimum: {df['salary_kes'].min():,.2f} KES")
print(f"Maximum: {df['salary_kes'].max():,.2f} KES")
print(f"Coefficient of variation: "
      f"{df['salary_kes'].std() / df['salary_kes'].mean():.3f}")

print("\nSTATISTICAL TESTS")

print(
    f"Performance vs salary: "
    f"rho = {performance_rho:.3f}, "
    f"p = {performance_p:.4f}"
)

print(
    f"Gender salary difference: "
    f"t = {gender_t:.3f}, "
    f"p = {gender_p:.4f}"
)

print(
    f"Age vs salary: "
    f"r = {age_r:.3f}, "
    f"p = {age_p:.4f}"
)

print(
    f"Department vs salary: "
    f"H = {department_h:.3f}, "
    f"p < 0.001" if department_p < 0.001
    else f"Department vs salary: H = {department_h:.3f}, p = {department_p:.4f}"
)

=== FINAL STATISTICAL SUMMARY ===

WORKFORCE
Employees: 420
Departments: 8
Job titles: 25

SALARY
Mean: 190,279.76 KES
Median: 166,000.00 KES
Standard deviation: 107,134.25 KES
Minimum: 41,000.00 KES
Maximum: 585,000.00 KES
Coefficient of variation: 0.563

STATISTICAL TESTS
Performance vs salary: rho = 0.033, p = 0.4962
Gender salary difference: t = 0.976, p = 0.3297
Age vs salary: r = 0.069, p = 0.1565
Department vs salary: H = 147.564, p < 0.001
